# Gamma Scalping Strategy Analysis
## Interactive visualization and comprehensive analysis using Plotly

This notebook provides a complete analysis of the gamma scalping strategy including:
- Strategy performance across different market periods
- Greeks evolution and hedging dynamics
- P&L attribution and risk analysis
- Interactive strategy comparison dashboard

In [20]:
# Setup and imports
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

# Set plotly theme
pio.templates.default = "plotly_white"

# Add parent directory to path
sys.path.append(str(Path().absolute().parent))

from data.data_loader import DataLoader
from simulation.gamma_scalping_simulator import GammaScalpingSimulator
from strategies.gamma_scalping import run_portfolio_backtest
from backtest.backtest_engine import StrategyConfig
from models.options_pricing import delta, gamma, theta, vega, bs_price, implied_volatility

print("📊 Gamma Scalping Analysis Notebook")
print("✅ Libraries loaded successfully!")

# Change to project root if in notebooks directory
import os
if os.getcwd().endswith('notebooks'):
    os.chdir('..')
    print(f"📁 Changed directory to: {os.getcwd()}")

📊 Gamma Scalping Analysis Notebook
✅ Libraries loaded successfully!


## 1. Data Loading and Overview
Load market data and get an overview of available options

In [33]:
# Load a comprehensive dataset
print("📥 Loading market data...")
start_date = pd.Timestamp('2023-01-01')
end_date = pd.Timestamp('2023-03-30')

# Initialize data loader
loader = DataLoader()
spot_data, options_dict = loader.create_dataset_dict(
    spot_symbol='BTCUSDT',
    underlying='BTC',
    start_date=start_date,
    end_date=end_date,
    option_filters={'min_volume': 0}  # Include all options
)

total_option_rows = sum(len(df) for df in options_dict.values()) if options_dict else 0
print(f"📈 Loaded {len(spot_data):,} spot prices")
print(f"📊 Loaded {total_option_rows:,} option data rows across {len(options_dict)} contracts")
print(f"📅 Date range: {spot_data['timestamp'].min().date()} to {spot_data['timestamp'].max().date()}")
print(f"💰 Spot price range: ${spot_data['close'].min():,.0f} - ${spot_data['close'].max():,.0f}")

if options_dict:
    print(f"🎯 Unique option contracts: {len(options_dict):,}")

# Create spot price chart
fig_spot = px.line(
    spot_data, 
    x='timestamp', 
    y='close',
    title='BTC Spot Price Evolution',
    labels={'close': 'Price (USD)', 'timestamp': 'Date'}
)
fig_spot.update_layout(height=400)
fig_spot.show()

📥 Loading market data...
📈 Loaded 2,112 spot prices
📊 Loaded 69,512 option data rows across 1321 contracts
📅 Date range: 2023-01-01 to 2023-03-30
💰 Spot price range: $16,513 - $28,647
🎯 Unique option contracts: 1,321


In [22]:
spot_data

,datetime,timestamp,open,high,low,close,volume,quote_volume,trades_count,taker_buy_volume,taker_buy_quote_volume
0,2023-01-01 00:00:00+00:00,2023-01-01 00:00:00,16541.77,16545.70,16508.39,16529.67,4364.83570,7.214629e+07,149854,2179.94772,3.603235e+07
1,2023-01-01 01:00:00+00:00,2023-01-01 01:00:00,16529.59,16556.80,16525.78,16551.47,3590.06669,5.937676e+07,126556,1730.24901,2.861742e+07
2,2023-01-01 02:00:00+00:00,2023-01-01 02:00:00,16551.47,16559.77,16538.14,16548.19,3318.84038,5.491945e+07,115398,1611.12302,2.666087e+07
3,2023-01-01 03:00:00+00:00,2023-01-01 03:00:00,16548.19,16548.19,16518.21,16533.04,4242.08050,7.012254e+07,137724,2096.09287,3.464904e+07
4,2023-01-01 04:00:00+00:00,2023-01-01 04:00:00,16533.04,16535.97,16511.92,16521.85,4285.00909,7.080264e+07,129535,2188.40175,3.615982e+07
...,...,...,...,...,...,...,...,...,...,...,...
2107,2023-03-29 20:00:00+00:00,2023-03-29 20:00:00,28419.49,28468.13,28356.23,28388.58,2226.26851,6.324659e+07,48405,1095.50999,3.112356e+07
2108,2023-03-29 21:00:00+00:00,2023-03-29 21:00:00,28388.58,28500.00,28290.35,28291.00,1916.30451,5.442486e+07,48825,972.23734,2.761713e+07
2109,2023-03-29 22:00:00+00:00,2023-03-29 22:00:00,28291.01,28429.28,28211.00,28389.41,2782.69442,7.878941e+07,58246,1455.01518,4.120070e+07
2110,2023-03-29 23:00:00+00:00,2023-03-29 23:00:00,28389.41,28456.84,28319.48,28348.60,1479.33184,4.200599e+07,37377,763.69239,2.168650e+07


In [23]:
options_dict["BTC-18MAR23-26250-P"]

,datetime,open,high,low,close,volume,trades,vwap,timestamp,underlying,expiry,strike,option_type,symbol
0,2023-03-17 06:00:00+00:00,0.0180,0.0190,0.0165,0.0165,3.8,5,0.016829,2023-03-17 06:00:00,BTC,2023-03-18 08:00:00,26250.0,put,BTC-18MAR23-26250-P
1,2023-03-17 07:00:00+00:00,0.0170,0.0170,0.0165,0.0165,7.7,2,0.016545,2023-03-17 07:00:00,BTC,2023-03-18 08:00:00,26250.0,put,BTC-18MAR23-26250-P
2,2023-03-17 08:00:00+00:00,0.0150,0.0150,0.0140,0.0140,1.1,2,0.014909,2023-03-17 08:00:00,BTC,2023-03-18 08:00:00,26250.0,put,BTC-18MAR23-26250-P
3,2023-03-17 09:00:00+00:00,0.0180,0.0180,0.0155,0.0155,7.8,6,0.017750,2023-03-17 09:00:00,BTC,2023-03-18 08:00:00,26250.0,put,BTC-18MAR23-26250-P
4,2023-03-17 10:00:00+00:00,0.0155,0.0155,0.0100,0.0100,23.7,29,0.011316,2023-03-17 10:00:00,BTC,2023-03-18 08:00:00,26250.0,put,BTC-18MAR23-26250-P
5,2023-03-17 11:00:00+00:00,0.0105,0.0115,0.0075,0.0075,14.4,9,0.010931,2023-03-17 11:00:00,BTC,2023-03-18 08:00:00,26250.0,put,BTC-18MAR23-26250-P
6,2023-03-17 12:00:00+00:00,0.0085,0.0105,0.0085,0.0095,3.0,5,0.009350,2023-03-17 12:00:00,BTC,2023-03-18 08:00:00,26250.0,put,BTC-18MAR23-26250-P
7,2023-03-17 13:00:00+00:00,0.0090,0.0115,0.0090,0.0115,36.4,4,0.011493,2023-03-17 13:00:00,BTC,2023-03-18 08:00:00,26250.0,put,BTC-18MAR23-26250-P
8,2023-03-17 14:00:00+00:00,0.0090,0.0130,0.0090,0.0120,11.5,9,0.011230,2023-03-17 14:00:00,BTC,2023-03-18 08:00:00,26250.0,put,BTC-18MAR23-26250-P
9,2023-03-17 15:00:00+00:00,0.0115,0.0120,0.0110,0.0110,7.3,6,0.011151,2023-03-17 15:00:00,BTC,2023-03-18 08:00:00,26250.0,put,BTC-18MAR23-26250-P


## 2. Strategy Performance Analysis
Run gamma scalping backtests and analyze performance

In [24]:
# Define strategy configurations
strategies = [
    StrategyConfig(
        name="Conservative (15%)",
        hedge_threshold=0.15,
        option_selection='atm',
        position_size=1.0
    ),
    StrategyConfig(
        name="Moderate (10%)",
        hedge_threshold=0.10,
        option_selection='atm',
        position_size=1.0
    ),
    StrategyConfig(
        name="Aggressive (5%)",
        hedge_threshold=0.05,
        option_selection='atm',
        position_size=1.0
    ),
    StrategyConfig(
        name="Very Aggressive (2.5%)",
        hedge_threshold=0.025,
        option_selection='atm',
        position_size=1.0
    )
]

print("🚀 Running strategy backtests...")
print("This may take a few minutes...")

# Run backtests using separate spot and per-option dict
results = run_portfolio_backtest(
    strategy_configs=strategies,
    spot_data=spot_data,
    options_dict=options_dict,
    max_options=15,  # Reasonable number for analysis
    target_dte=30,
    moneyness_filter='ATM'
)

print(f"✅ Backtests completed! Analyzed {len(results)} strategy-option combinations")

🚀 Running strategy backtests...
This may take a few minutes...
GAMMA SCALPING BACKTEST
Period: 2023-01-01 to 2023-03-30
Target DTE: 30 days
Moneyness filter: ATM

Loaded spot rows: 2,112 | options rows total: 69,512

Analyzing option chain...
Selected 15 options for testing
Strike range: 21500 - 25000
DTE range: 20 - 34 days

--- Testing strategy: Conservative (15%) ---
Selected 15 options for testing
Strike range: 21500 - 25000
DTE range: 20 - 34 days

--- Testing strategy: Conservative (15%) ---


Options: 100%|██████████| 15/15 [00:02<00:00,  5.41it/s]



Results for Conservative (15%):
  Total P&L: $136,069.84
  Average P&L per option: $9,071.32
  Win rate: 66.7%
  Options traded: 15

--- Testing strategy: Moderate (10%) ---


Options: 100%|██████████| 15/15 [00:02<00:00,  5.39it/s]



Results for Moderate (10%):
  Total P&L: $139,109.27
  Average P&L per option: $9,273.95
  Win rate: 66.7%
  Options traded: 15

--- Testing strategy: Aggressive (5%) ---


Options: 100%|██████████| 15/15 [00:02<00:00,  5.44it/s]



Results for Aggressive (5%):
  Total P&L: $141,158.85
  Average P&L per option: $9,410.59
  Win rate: 66.7%
  Options traded: 15

--- Testing strategy: Very Aggressive (2.5%) ---


Options: 100%|██████████| 15/15 [00:02<00:00,  5.10it/s]


Results for Very Aggressive (2.5%):
  Total P&L: $142,999.46
  Average P&L per option: $9,533.30
  Win rate: 66.7%
  Options traded: 15

SUMMARY BY STRATEGY
                                        total_pnl                     \
                                              sum     mean       std   
strategy               hedge_threshold                                 
Aggressive (5%)        0.050            141158.85  9410.59  15700.81   
Conservative (15%)     0.150            136069.84  9071.32  15287.85   
Moderate (10%)         0.100            139109.27  9273.95  15555.11   
Very Aggressive (2.5%) 0.025            142999.46  9533.30  15795.91   

                                       option_pnl  hedge_pnl commission_cost  \
                                              sum        sum             sum   
strategy               hedge_threshold                                         
Aggressive (5%)        0.050             16633.56  128215.77         1230.16   
Conservative (15%

In [25]:
# Create performance dashboard
if not results.empty:
    # Aggregate results by strategy
    strategy_summary = results.groupby('strategy').agg({
        'total_pnl': 'sum',
        'option_pnl': 'sum', 
        'hedge_pnl': 'sum',
        'commission_cost': 'sum',
        'slippage_cost': 'sum',
        'num_hedges': 'mean',
        'gamma_pnl': 'sum',
        'theta_cost': 'sum'
    }).reset_index()
    
    strategy_summary['win_rate'] = results.groupby('strategy')['total_pnl'].apply(lambda x: (x > 0).mean()).values
    strategy_summary['num_options'] = results.groupby('strategy').size().values
    
    # Create subplot dashboard
    fig = make_subplots(
        rows=2, cols=3,
        subplot_titles=[
            'Total P&L by Strategy',
            'Win Rate vs Average Hedges',
            'P&L Attribution',
            'Transaction Costs',
            'Gamma vs Theta Effects',
            'Strategy Comparison'
        ],
        specs=[[{'type': 'bar'}, {'type': 'scatter'}, {'type': 'bar'}],
               [{'type': 'bar'}, {'type': 'scatter'}, {'type': 'bar'}]]
    )
    
    # 1. Total P&L by Strategy
    colors = ['green' if x > 0 else 'red' for x in strategy_summary['total_pnl']]
    fig.add_trace(
        go.Bar(
            x=strategy_summary['strategy'],
            y=strategy_summary['total_pnl'],
            name='Total P&L',
            marker_color=colors,
            showlegend=False
        ),
        row=1, col=1
    )
    
    # 2. Win Rate vs Avg Hedges
    fig.add_trace(
        go.Scatter(
            x=strategy_summary['num_hedges'],
            y=strategy_summary['win_rate'] * 100,
            mode='markers+text',
            text=strategy_summary['strategy'],
            textposition='top center',
            marker=dict(size=strategy_summary['total_pnl']/1000 + 10, opacity=0.7),
            name='Strategies',
            showlegend=False
        ),
        row=1, col=2
    )
    
    # 3. P&L Attribution
    fig.add_trace(
        go.Bar(
            x=strategy_summary['strategy'],
            y=strategy_summary['option_pnl'],
            name='Option P&L',
            marker_color='lightblue'
        ),
        row=1, col=3
    )
    fig.add_trace(
        go.Bar(
            x=strategy_summary['strategy'],
            y=strategy_summary['hedge_pnl'],
            name='Hedge P&L',
            marker_color='lightcoral'
        ),
        row=1, col=3
    )
    
    # 4. Transaction Costs
    total_costs = strategy_summary['commission_cost'] + strategy_summary['slippage_cost']
    fig.add_trace(
        go.Bar(
            x=strategy_summary['strategy'],
            y=total_costs,
            name='Transaction Costs',
            marker_color='orange',
            showlegend=False
        ),
        row=2, col=1
    )
    
    # 5. Gamma vs Theta Effects  
    fig.add_trace(
        go.Scatter(
            x=strategy_summary['gamma_pnl'],
            y=strategy_summary['theta_cost'],
            mode='markers+text',
            text=strategy_summary['strategy'],
            textposition='top center',
            marker=dict(size=12, opacity=0.7),
            name='Gamma vs Theta',
            showlegend=False
        ),
        row=2, col=2
    )
    
    # 6. Strategy Ranking
    strategy_summary_sorted = strategy_summary.sort_values('total_pnl', ascending=True)
    fig.add_trace(
        go.Bar(
            y=strategy_summary_sorted['strategy'],
            x=strategy_summary_sorted['total_pnl'],
            orientation='h',
            name='Strategy Ranking',
            marker_color=['green' if x > 0 else 'red' for x in strategy_summary_sorted['total_pnl']],
            showlegend=False
        ),
        row=2, col=3
    )
    
    # Update layout
    fig.update_layout(
        height=800,
        title_text="Gamma Scalping Strategy Performance Dashboard",
        title_x=0.5,
        showlegend=True
    )
    
    # Update axis labels
    fig.update_xaxes(title_text="Strategy", row=1, col=1)
    fig.update_yaxes(title_text="P&L ($)", row=1, col=1)
    
    fig.update_xaxes(title_text="Average Hedges", row=1, col=2)
    fig.update_yaxes(title_text="Win Rate (%)", row=1, col=2)
    
    fig.update_xaxes(title_text="Strategy", row=1, col=3)
    fig.update_yaxes(title_text="P&L ($)", row=1, col=3)
    
    fig.update_xaxes(title_text="Strategy", row=2, col=1)
    fig.update_yaxes(title_text="Cost ($)", row=2, col=1)
    
    fig.update_xaxes(title_text="Gamma P&L ($)", row=2, col=2)
    fig.update_yaxes(title_text="Theta Cost ($)", row=2, col=2)
    
    fig.update_xaxes(title_text="Total P&L ($)", row=2, col=3)
    fig.update_yaxes(title_text="Strategy", row=2, col=3)
    
    fig.show()
    
    # Print summary statistics
    print("\n📊 Strategy Performance Summary:")
    print("=" * 50)
    for _, row in strategy_summary.iterrows():
        print(f"\n{row['strategy']}:")
        print(f"  💰 Total P&L: ${row['total_pnl']:,.2f}")
        print(f"  🎯 Win Rate: {row['win_rate']:.1%}")
        print(f"  📈 Options Traded: {row['num_options']}")
        print(f"  🔄 Avg Hedges: {row['num_hedges']:.1f}")

else:
    print("❌ No backtest results available")


📊 Strategy Performance Summary:

Aggressive (5%):
  💰 Total P&L: $141,158.85
  🎯 Win Rate: 66.7%
  📈 Options Traded: 15
  🔄 Avg Hedges: 18.4

Conservative (15%):
  💰 Total P&L: $136,069.84
  🎯 Win Rate: 66.7%
  📈 Options Traded: 15
  🔄 Avg Hedges: 12.7

Moderate (10%):
  💰 Total P&L: $139,109.27
  🎯 Win Rate: 66.7%
  📈 Options Traded: 15
  🔄 Avg Hedges: 14.6

Very Aggressive (2.5%):
  💰 Total P&L: $142,999.46
  🎯 Win Rate: 66.7%
  📈 Options Traded: 15
  🔄 Avg Hedges: 24.5


## 3. Greeks Evolution Analysis
Analyze how option Greeks evolve over time and their impact on P&L

In [26]:
def analyze_option_greeks(spot_data, strike=30000, expiry_days=30, iv=0.8):
    """Analyze Greeks evolution for a specific option"""
    
    expiry = spot_data['timestamp'].min() + pd.Timedelta(days=expiry_days)
    risk_free_rate = 0.01
    
    greeks_data = []
    
    for _, row in spot_data.iterrows():
        spot = row['close']
        timestamp = row['timestamp']
        
        # Calculate time to expiry
        tte = max(0, (expiry - timestamp).total_seconds() / (365 * 24 * 3600))
        
        if tte > 0:
            # Calculate Greeks for call option
            call_delta = delta(spot, strike, tte, risk_free_rate, iv, 'call')
            call_gamma = gamma(spot, strike, tte, risk_free_rate, iv)
            call_theta = theta(spot, strike, tte, risk_free_rate, iv, 'call')
            call_vega = vega(spot, strike, tte, risk_free_rate, iv)
            call_price = bs_price(spot, strike, tte, risk_free_rate, iv, 'call')
            
            greeks_data.append({
                'timestamp': timestamp,
                'spot': spot,
                'strike': strike,
                'tte': tte * 365,  # Days
                'delta': call_delta,
                'gamma': call_gamma,
                'theta': call_theta / 365,  # Daily
                'vega': call_vega / 100,  # Per 1% IV
                'option_price': call_price,
                'moneyness': spot / strike,
                'intrinsic_value': max(0, spot - strike)
            })
    
    return pd.DataFrame(greeks_data)

# Analyze Greeks for a representative option
print("📈 Analyzing Greeks evolution...")
initial_spot = spot_data['close'].iloc[0]
atm_strike = round(initial_spot / 1000) * 1000  # Round to nearest 1000

greeks_df = analyze_option_greeks(
    spot_data=spot_data.iloc[:1000],  # First ~40 days of data
    strike=atm_strike,
    expiry_days=30,
    iv=0.8
)

print(f"🎯 Analyzing ATM option with strike ${atm_strike:,}")
print(f"📊 Generated {len(greeks_df)} data points")

📈 Analyzing Greeks evolution...
🎯 Analyzing ATM option with strike $17,000
📊 Generated 720 data points


In [27]:
# Create comprehensive Greeks visualization
fig_greeks = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Spot Price and Option Value',
        'Delta Evolution',
        'Gamma (Sensitivity)',
        'Theta Decay',
    ],
    specs=[[{'secondary_y': True}, {'type': 'scatter'}],
           [{'type': 'scatter'}, {'type': 'scatter'}]]
)

# 1. Spot Price and Option Value
fig_greeks.add_trace(
    go.Scatter(
        x=greeks_df['timestamp'],
        y=greeks_df['spot'],
        name='Spot Price',
        line=dict(color='black', width=2)
    ),
    row=1, col=1, secondary_y=False
)

fig_greeks.add_trace(
    go.Scatter(
        x=greeks_df['timestamp'],
        y=greeks_df['option_price'],
        name='Option Price',
        line=dict(color='blue', width=2)
    ),
    row=1, col=1, secondary_y=True
)

# 2. Delta Evolution
fig_greeks.add_trace(
    go.Scatter(
        x=greeks_df['timestamp'],
        y=greeks_df['delta'],
        name='Delta',
        line=dict(color='green', width=2)
    ),
    row=1, col=2
)
fig_greeks.add_hline(y=0.5, line_dash="dash", line_color="red", 
                    annotation_text="ATM Delta", row=1, col=2)

# 3. Gamma
fig_greeks.add_trace(
    go.Scatter(
        x=greeks_df['timestamp'],
        y=greeks_df['gamma'] * 10000,  # Scale for readability
        name='Gamma (×10k)',
        line=dict(color='orange', width=2),
        fill='tonexty'
    ),
    row=2, col=1
)

# 4. Theta
fig_greeks.add_trace(
    go.Scatter(
        x=greeks_df['timestamp'],
        y=greeks_df['theta'],
        name='Daily Theta',
        line=dict(color='red', width=2),
        fill='tozeroy'
    ),
    row=2, col=2
)

# Update layout
fig_greeks.update_layout(
    height=800,
    title_text=f"Option Greeks Evolution Analysis - ATM Call (${atm_strike:,} Strike)",
    title_x=0.5,
    showlegend=True
)

# Update axis labels
fig_greeks.update_yaxes(title_text="Spot Price ($)", secondary_y=False, row=1, col=1)
fig_greeks.update_yaxes(title_text="Option Price ($)", secondary_y=True, row=1, col=1)
fig_greeks.update_yaxes(title_text="Delta", row=1, col=2)
fig_greeks.update_yaxes(title_text="Gamma (×10k)", row=2, col=1)
fig_greeks.update_yaxes(title_text="Theta ($)", row=2, col=2)
fig_greeks.update_xaxes(title_text="Moneyness (S/K)", row=3, col=1)
fig_greeks.update_yaxes(title_text="Delta", row=3, col=1)
fig_greeks.update_xaxes(title_text="Days to Expiry", row=3, col=2)
fig_greeks.update_yaxes(title_text="Option Price ($)", row=3, col=2)

fig_greeks.show()

# Print Greeks summary
print("\n🎯 Greeks Analysis Summary:")
print("=" * 40)
print(f"📊 Delta range: {greeks_df['delta'].min():.3f} to {greeks_df['delta'].max():.3f}")
print(f"📊 Max Gamma: {greeks_df['gamma'].max():.6f}")
print(f"📊 Avg daily Theta: ${greeks_df['theta'].mean():.2f}")
print(f"📊 Final option value: ${greeks_df['option_price'].iloc[-1]:.2f}")


🎯 Greeks Analysis Summary:
📊 Delta range: 0.496 to 1.000
📊 Max Gamma: 0.000122
📊 Avg daily Theta: $-16.99
📊 Final option value: $5826.17


## 4. Hedging Activity Analysis
Simulate and visualize delta hedging activity with different thresholds

In [28]:
def simulate_hedging_activity(spot_data, strike, expiry_days, iv, hedge_threshold=0.1):
    """Simulate detailed hedging activity"""
    
    expiry = spot_data['timestamp'].min() + pd.Timedelta(days=expiry_days)
    hedge_position = 0
    last_delta = 0
    
    hedging_data = []
    
    for _, row in spot_data.iterrows():
        spot = row['close']
        timestamp = row['timestamp']
        
        tte = max(0, (expiry - timestamp).total_seconds() / (365 * 24 * 3600))
        
        if tte > 0:
            current_delta = delta(spot, strike, tte, 0.01, iv, 'call')
            current_gamma = gamma(spot, strike, tte, 0.01, iv)
            
            # Check if hedging is needed using delta-band method
            delta_change = abs(current_delta - last_delta)
            needs_hedge = delta_change > hedge_threshold
            
            hedge_trade = 0
            if needs_hedge:
                # Calculate hedge trade size
                target_hedge = -current_delta  # Hedge opposite to delta
                hedge_trade = target_hedge - hedge_position
                hedge_position = target_hedge
                last_delta = current_delta
            
            # Calculate net exposure
            net_delta = current_delta + hedge_position
            
            hedging_data.append({
                'timestamp': timestamp,
                'spot': spot,
                'option_delta': current_delta,
                'hedge_position': hedge_position,
                'net_delta': net_delta,
                'gamma': current_gamma,
                'hedge_trade': hedge_trade,
                'needs_hedge': needs_hedge,
                'threshold': hedge_threshold
            })
    
    return pd.DataFrame(hedging_data)

# Simulate different hedging strategies
print("🔄 Simulating hedging activities...")

hedge_scenarios = [
    {'name': 'Conservative (15%)', 'threshold': 0.15, 'color': 'blue'},
    {'name': 'Moderate (10%)', 'threshold': 0.10, 'color': 'green'},
    {'name': 'Aggressive (5%)', 'threshold': 0.05, 'color': 'orange'},
    {'name': 'Very Aggressive (2.5%)', 'threshold': 0.025, 'color': 'red'}
]

hedge_results = {}
for scenario in hedge_scenarios:
    hedge_results[scenario['name']] = simulate_hedging_activity(
        spot_data=spot_data.iloc[:500],  # First ~20 days
        strike=atm_strike,
        expiry_days=30,
        iv=0.8,
        hedge_threshold=scenario['threshold']
    )
    
    trades = hedge_results[scenario['name']][hedge_results[scenario['name']]['hedge_trade'] != 0]
    print(f"📊 {scenario['name']}: {len(trades)} hedge trades")

print(f"✅ Hedging simulation completed!")

🔄 Simulating hedging activities...
📊 Conservative (15%): 4 hedge trades
📊 Moderate (10%): 4 hedge trades
📊 Aggressive (5%): 8 hedge trades
📊 Very Aggressive (2.5%): 14 hedge trades
✅ Hedging simulation completed!


In [29]:
# Create comprehensive hedging visualization
fig_hedge = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Delta Evolution and Hedge Positions',
        'Net Delta Exposure',
        'Hedge Trade Frequency',
        'Cumulative Hedge Volume'
    ]
)

# Colors for different strategies
colors = ['blue', 'green', 'orange', 'red']

# 1. Delta Evolution and Hedge Positions
base_df = hedge_results['Moderate (10%)']
fig_hedge.add_trace(
    go.Scatter(
        x=base_df['timestamp'],
        y=base_df['option_delta'],
        name='Option Delta',
        line=dict(color='black', width=3)
    ),
    row=1, col=1
)

for i, (name, df) in enumerate(hedge_results.items()):
    fig_hedge.add_trace(
        go.Scatter(
            x=df['timestamp'],
            y=df['hedge_position'],
            name=f'Hedge: {name}',
            line=dict(color=colors[i], width=2, dash='dash')
        ),
        row=1, col=1
    )

# 2. Net Delta Exposure
for i, (name, df) in enumerate(hedge_results.items()):
    fig_hedge.add_trace(
        go.Scatter(
            x=df['timestamp'],
            y=df['net_delta'],
            name=f'Net Δ: {name}',
            line=dict(color=colors[i], width=2),
            fill='tonexty' if i == 0 else None,
            showlegend=False
        ),
        row=1, col=2
    )

fig_hedge.add_hline(y=0, line_dash="solid", line_color="black", row=1, col=2)

# 3. Hedge Trade Frequency (bar chart)
trade_counts = []
strategy_names = []
avg_trade_sizes = []

for name, df in hedge_results.items():
    trades = df[df['hedge_trade'] != 0]
    trade_counts.append(len(trades))
    strategy_names.append(name.split(' (')[0])  # Shorten name
    avg_trade_sizes.append(abs(trades['hedge_trade']).mean() if len(trades) > 0 else 0)

fig_hedge.add_trace(
    go.Bar(
        x=strategy_names,
        y=trade_counts,
        name='Trade Count',
        marker_color=colors,
        showlegend=False
    ),
    row=2, col=1
)

# 4. Cumulative Hedge Volume
for i, (name, df) in enumerate(hedge_results.items()):
    cumulative_volume = abs(df['hedge_trade']).cumsum()
    fig_hedge.add_trace(
        go.Scatter(
            x=df['timestamp'],
            y=cumulative_volume,
            name=f'Vol: {name}',
            line=dict(color=colors[i], width=2),
            showlegend=False
        ),
        row=2, col=2
    )

# Update layout
fig_hedge.update_layout(
    height=800,
    title_text="Delta Hedging Activity Analysis",
    title_x=0.5,
    showlegend=True
)

# Update axis labels
fig_hedge.update_yaxes(title_text="Delta", row=1, col=1)
fig_hedge.update_yaxes(title_text="Net Delta", row=1, col=2)
fig_hedge.update_yaxes(title_text="Number of Trades", row=2, col=1)
fig_hedge.update_yaxes(title_text="Cumulative Volume", row=2, col=2)

fig_hedge.show()

# Print hedging statistics
print("\n🔄 Hedging Activity Summary:")
print("=" * 50)
for i, (name, df) in enumerate(hedge_results.items()):
    trades = df[df['hedge_trade'] != 0]
    max_net_delta = abs(df['net_delta']).max()
    total_volume = abs(df['hedge_trade']).sum()
    
    print(f"\n{name}:")
    print(f"  🔢 Total Trades: {len(trades)}")
    print(f"  📊 Avg Trade Size: {avg_trade_sizes[i]:.4f}")
    print(f"  📈 Max Net Delta: {max_net_delta:.4f}")
    print(f"  📦 Total Volume: {total_volume:.2f}")


🔄 Hedging Activity Summary:

Conservative (15%):
  🔢 Total Trades: 4
  📊 Avg Trade Size: 0.2400
  📈 Max Net Delta: 0.1475
  📦 Total Volume: 0.96

Moderate (10%):
  🔢 Total Trades: 4
  📊 Avg Trade Size: 0.2251
  📈 Max Net Delta: 0.0997
  📦 Total Volume: 0.90

Aggressive (5%):
  🔢 Total Trades: 8
  📊 Avg Trade Size: 0.1189
  📈 Max Net Delta: 0.0496
  📦 Total Volume: 0.95

Very Aggressive (2.5%):
  🔢 Total Trades: 14
  📊 Avg Trade Size: 0.0741
  📈 Max Net Delta: 0.0247
  📦 Total Volume: 1.04


## 5. P&L Attribution Analysis
Detailed breakdown of gamma scalping P&L sources

In [30]:
# Get detailed P&L data from a successful strategy result
if not results.empty:
    # Get the best performing strategy for detailed analysis
    best_strategy = results.nlargest(1, 'total_pnl').iloc[0]
    print(f"🎯 Analyzing detailed P&L for: {best_strategy['strategy']}")
    
    # Create P&L visualization using the actual results data
    fig_pnl = make_subplots(
        rows=2, cols=2,
        subplot_titles=[
            'P&L Components Distribution',
            'Strategy Performance Comparison',
            'Risk-Return Profile', 
            'Transaction Cost Analysis'
        ],
        specs=[[{'type': 'histogram'}, {'type': 'bar'}],
               [{'type': 'scatter'}, {'type': 'pie'}]]
    )
    
    # 1. P&L Distribution
    fig_pnl.add_trace(
        go.Histogram(
            x=results['total_pnl'],
            name='Total P&L Distribution',
            nbinsx=20,
            marker_color='lightblue',
            opacity=0.7
        ),
        row=1, col=1
    )
    
    # Add mean line
    mean_pnl = results['total_pnl'].mean()
    fig_pnl.add_vline(x=mean_pnl, line_dash="dash", line_color="red", 
                     annotation_text=f"Mean: ${mean_pnl:.0f}", row=1, col=1)
    
    # 2. Strategy Performance Comparison
    strategy_perf = results.groupby('strategy').agg({
        'total_pnl': 'sum',
        'option_pnl': 'sum',
        'hedge_pnl': 'sum'
    }).reset_index()
    
    fig_pnl.add_trace(
        go.Bar(
            x=strategy_perf['strategy'],
            y=strategy_perf['option_pnl'],
            name='Option P&L',
            marker_color='lightcoral'
        ),
        row=1, col=2
    )
    
    fig_pnl.add_trace(
        go.Bar(
            x=strategy_perf['strategy'],
            y=strategy_perf['hedge_pnl'],
            name='Hedge P&L',
            marker_color='lightgreen'
        ),
        row=1, col=2
    )
    
    # 3. Risk-Return Profile
    strategy_stats = results.groupby('strategy').agg({
        'total_pnl': ['mean', 'std']
    }).reset_index()
    strategy_stats.columns = ['strategy', 'avg_pnl', 'pnl_std']
    
    fig_pnl.add_trace(
        go.Scatter(
            x=strategy_stats['pnl_std'],
            y=strategy_stats['avg_pnl'],
            mode='markers+text',
            text=strategy_stats['strategy'],
            textposition='top center',
            marker=dict(size=15, opacity=0.7),
            name='Risk-Return',
            showlegend=False
        ),
        row=2, col=1
    )
    
    # 4. Transaction Cost Breakdown
    total_commission = results['commission_cost'].sum()
    total_slippage = results['slippage_cost'].sum()
    total_pnl_gross = results['total_pnl'].sum() + total_commission + total_slippage
    
    fig_pnl.add_trace(
        go.Pie(
            labels=['Gross P&L', 'Commission', 'Slippage'],
            values=[total_pnl_gross, total_commission, total_slippage],
            name='Cost Breakdown'
        ),
        row=2, col=2
    )
    
    # Update layout
    fig_pnl.update_layout(
        height=800,
        title_text="P&L Attribution and Risk Analysis",
        title_x=0.5,
        showlegend=True
    )
    
    # Update axis labels
    fig_pnl.update_xaxes(title_text="P&L ($)", row=1, col=1)
    fig_pnl.update_yaxes(title_text="Frequency", row=1, col=1)
    fig_pnl.update_yaxes(title_text="P&L ($)", row=1, col=2)
    fig_pnl.update_xaxes(title_text="Risk (Std Dev)", row=2, col=1)
    fig_pnl.update_yaxes(title_text="Return (Mean P&L)", row=2, col=1)
    
    fig_pnl.show()
    
    # Print detailed P&L analysis
    print("\n💰 P&L Attribution Summary:")
    print("=" * 50)
    print(f"📊 Total Strategies Tested: {len(results['strategy'].unique())}")
    print(f"📊 Total Option Contracts: {len(results)}")
    print(f"💰 Total P&L: ${results['total_pnl'].sum():,.2f}")
    print(f"📈 Option P&L: ${results['option_pnl'].sum():,.2f}")
    print(f"🔄 Hedge P&L: ${results['hedge_pnl'].sum():,.2f}")
    print(f"💸 Total Commissions: ${results['commission_cost'].sum():,.2f}")
    print(f"💸 Total Slippage: ${results['slippage_cost'].sum():,.2f}")
    print(f"🎯 Win Rate: {(results['total_pnl'] > 0).mean():.1%}")
    print(f"📊 Best Single Trade: ${results['total_pnl'].max():,.2f}")
    print(f"📊 Worst Single Trade: ${results['total_pnl'].min():,.2f}")

else:
    print("❌ No results available for P&L analysis")

🎯 Analyzing detailed P&L for: Aggressive (5%)



💰 P&L Attribution Summary:
📊 Total Strategies Tested: 4
📊 Total Option Contracts: 60
💰 Total P&L: $559,337.41
📈 Option P&L: $66,534.25
🔄 Hedge P&L: $507,320.62
💸 Total Commissions: $4,839.15
💸 Total Slippage: $9,678.30
🎯 Win Rate: 66.7%
📊 Best Single Trade: $29,635.04
📊 Worst Single Trade: $-21,204.49


## 6. Volatility Analysis
Analyze realized vs implied volatility relationships

In [31]:
def calculate_realized_volatility(spot_data, windows=[24, 168, 720]):
    """Calculate realized volatility over different time windows"""
    
    # Calculate log returns
    spot_data = spot_data.copy()
    spot_data['log_return'] = np.log(spot_data['close'] / spot_data['close'].shift(1))
    
    vol_data = spot_data[['timestamp', 'close', 'log_return']].copy()
    
    # Calculate rolling volatility for different windows
    for window in windows:
        # Annualized volatility (assuming hourly data)
        vol_data[f'rv_{window}h'] = (
            vol_data['log_return'].rolling(window=window).std() * np.sqrt(24 * 365)
        )
    
    return vol_data.dropna()

# Calculate realized volatility
print("📊 Calculating realized volatility...")
vol_data = calculate_realized_volatility(spot_data)

# Get implied volatility from options dict if available
iv_data = []
if options_dict:
    print("📈 Extracting implied volatility from options...")
    
    # Concatenate a manageable subset of options for sampling
    concat_limit = 10  # number of option symbols to sample
    sampled = []
    for i, (sym, df) in enumerate(options_dict.items()):
        if i >= concat_limit:
            break
        d = df[['timestamp','close','strike','expiry','option_type']].copy()
        d['symbol'] = sym
        sampled.append(d)
    options_sample = pd.concat(sampled, ignore_index=True) if sampled else pd.DataFrame()
    
    # Sample every 10th timestamp for performance
    sample_timestamps = spot_data['timestamp'].iloc[::10]
    
    for timestamp in sample_timestamps[:100]:  # Limit for performance
        spot_price = spot_data[spot_data['timestamp'] == timestamp]['close']
        if spot_price.empty:
            continue
        spot_price = spot_price.iloc[0]
        
        # Find ATM options at this timestamp
        opts_at_time = options_sample[options_sample['timestamp'] == timestamp]
        if opts_at_time.empty:
            continue
            
        opts_at_time = opts_at_time.copy()
        opts_at_time['moneyness'] = abs(opts_at_time['strike'] - spot_price) / spot_price
        
        # Get closest to ATM
        atm_option = opts_at_time.nsmallest(1, 'moneyness')
        
        if not atm_option.empty:
            opt = atm_option.iloc[0]
            tte = (opt['expiry'] - timestamp).total_seconds() / (365 * 24 * 3600)
            
            if tte > 0:
                try:
                    iv = implied_volatility(
                        opt['close'], spot_price, opt['strike'],
                        tte, 0.01, opt['option_type']
                    )
                    
                    if not np.isnan(iv) and 0.1 < iv < 3.0:  # Reasonable IV range
                        iv_data.append({
                            'timestamp': timestamp,
                            'spot': spot_price,
                            'strike': opt['strike'],
                            'iv': iv,
                            'dte': tte * 365,
                            'option_type': opt['option_type']
                        })
                except:
                    continue

iv_df = pd.DataFrame(iv_data)
print(f"📊 Extracted {len(iv_df)} IV data points")

📊 Calculating realized volatility...
📈 Extracting implied volatility from options...
📊 Extracted 30 IV data points
📊 Extracted 30 IV data points


In [32]:
# Create volatility analysis visualization
fig_vol = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Realized Volatility (Multiple Windows)',
        'Implied vs Realized Volatility',
        'Volatility Premium Over Time',
        'Vol Distribution Analysis'
    ],
    specs=[[{'type': 'scatter'}, {'type': 'scatter'}],
           [{'type': 'scatter'}, {'type': 'histogram'}]]
)

# 1. Realized Volatility (Multiple Windows)
colors_rv = ['blue', 'green', 'red']
windows = [24, 168, 720]  # 1 day, 1 week, 1 month
window_names = ['1D', '1W', '1M']

for i, (window, name) in enumerate(zip(windows, window_names)):
    fig_vol.add_trace(
        go.Scatter(
            x=vol_data['timestamp'],
            y=vol_data[f'rv_{window}h'] * 100,
            name=f'RV {name}',
            line=dict(color=colors_rv[i], width=2)
        ),
        row=1, col=1
    )

# 2. Implied vs Realized Volatility
if not iv_df.empty:
    # Align IV and RV data
    merged_vol = pd.merge_asof(
        iv_df.sort_values('timestamp'),
        vol_data[['timestamp', 'rv_24h']].sort_values('timestamp'),
        on='timestamp',
        direction='nearest'
    )
    
    fig_vol.add_trace(
        go.Scatter(
            x=merged_vol['timestamp'],
            y=merged_vol['rv_24h'] * 100,
            name='Realized Vol (1D)',
            line=dict(color='blue', width=2)
        ),
        row=1, col=2
    )
    
    fig_vol.add_trace(
        go.Scatter(
            x=merged_vol['timestamp'],
            y=merged_vol['iv'] * 100,
            name='Implied Vol (ATM)',
            mode='markers+lines',
            marker=dict(size=6),
            line=dict(color='red', width=2)
        ),
        row=1, col=2
    )
    
    # 3. Volatility Premium
    if 'rv_24h' in merged_vol.columns:
        vol_premium = (merged_vol['iv'] - merged_vol['rv_24h']) * 100
        
        fig_vol.add_trace(
            go.Scatter(
                x=merged_vol['timestamp'],
                y=vol_premium,
                name='Vol Premium (IV - RV)',
                fill='tonexty',
                line=dict(color='purple', width=2),
                showlegend=False
            ),
            row=2, col=1
        )
        
        fig_vol.add_hline(y=0, line_dash="dash", line_color="black", row=2, col=1)
    
    # 4. Volatility Distribution
    fig_vol.add_trace(
        go.Histogram(
            x=merged_vol['rv_24h'] * 100,
            name='Realized Vol',
            opacity=0.7,
            marker_color='blue',
            nbinsx=30
        ),
        row=2, col=2
    )
    
    fig_vol.add_trace(
        go.Histogram(
            x=merged_vol['iv'] * 100,
            name='Implied Vol',
            opacity=0.7,
            marker_color='red',
            nbinsx=30
        ),
        row=2, col=2
    )

else:
    # If no IV data, show RV distribution only
    fig_vol.add_trace(
        go.Histogram(
            x=vol_data['rv_24h'] * 100,
            name='Realized Vol (1D)',
            opacity=0.7,
            marker_color='blue',
            nbinsx=30
        ),
        row=2, col=2
    )

# Update layout
fig_vol.update_layout(
    height=800,
    title_text="Volatility Analysis Dashboard",
    title_x=0.5,
    showlegend=True
)

# Update axis labels
fig_vol.update_yaxes(title_text="Volatility (%)", row=1, col=1)
fig_vol.update_yaxes(title_text="Volatility (%)", row=1, col=2)
fig_vol.update_yaxes(title_text="Premium (%)", row=2, col=1)
fig_vol.update_xaxes(title_text="Volatility (%)", row=2, col=2)
fig_vol.update_yaxes(title_text="Frequency", row=2, col=2)

fig_vol.show()

# Print volatility statistics
print("\n📊 Volatility Analysis Summary:")
print("=" * 50)
print(f"📈 Average RV (1D): {vol_data['rv_24h'].mean() * 100:.1f}%")
print(f"📈 Average RV (1W): {vol_data['rv_168h'].mean() * 100:.1f}%")
print(f"📈 Average RV (1M): {vol_data['rv_720h'].mean() * 100:.1f}%")
print(f"📊 RV Range: {vol_data['rv_24h'].min() * 100:.1f}% - {vol_data['rv_24h'].max() * 100:.1f}%")

if not iv_df.empty and 'rv_24h' in merged_vol.columns:
    avg_iv = merged_vol['iv'].mean() * 100
    avg_rv = merged_vol['rv_24h'].mean() * 100
    avg_premium = avg_iv - avg_rv
    
    print(f"📈 Average IV: {avg_iv:.1f}%")
    print(f"💰 Average Vol Premium: {avg_premium:.1f}%")
    print(f"🎯 Vol Premium positive: {(vol_premium > 0).mean():.1%} of the time")


📊 Volatility Analysis Summary:
📈 Average RV (1D): 50.2%
📈 Average RV (1W): 54.0%
📈 Average RV (1M): 51.3%
📊 RV Range: 12.7% - 134.5%
📈 Average IV: 16.9%
💰 Average Vol Premium: -25.9%
🎯 Vol Premium positive: 3.3% of the time
